# Validation A.1 — soda-lime hemisphere workflow

This notebook validates the soda-lime hemisphere row from Fig. 3(b) blue / Table 1 of Silva-Oelker and Jaramillo-Fernandez (2022). It rebuilds the normal-incidence optical spectrum with S4 and compares it against the published optical columns, then reads the published reduced hemispherical spectrum to compare the thermal and PV parameters against Table 1.

**Learning goals:** compare optical band averages against the published spectrum; compare thermal/PV outputs against Table 1; keep the normal-incidence optical check separate from the reduced-spectrum thermal/PV check.

## 1. Prepare the temporary Colab runtime

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shlex
import subprocess
import sys

from IPython.display import Image, Markdown, display

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("Open this notebook in Google Colab before running setup.")

PROJECT_DIR = Path("/content/radcoolpv-py")

def run_command(args: list[str], cwd: Path | None = None, capture: bool = False):
    print("$", shlex.join(args))
    return subprocess.run(
        args, cwd=cwd, check=True, text=True,
        capture_output=capture,
    )

if not PROJECT_DIR.exists():
    run_command([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/gsilvaoelker/radcoolpv-py.git",
        str(PROJECT_DIR),
    ])

run_command([
    sys.executable, "-m", "pip", "install", "--quiet", "--editable", ".",
], cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)

print("Repository:", PROJECT_DIR)

## 2. Rebuild and compare the optical spectrum

The live comparison compiles the supported S4 revision if needed, runs the paper-stated soda-lime geometry at normal incidence, and prints band averages against the published Fig. 3(b) normal-incidence columns.

In [ ]:
import importlib
import importlib.util

S4_DIR = Path("/content/S4")
S4_COMMIT = "9569f5e555b967a4324eb1ea593d0f9f40761a61"

def install_s4() -> None:
    """Build the supported phoebe-p/S4 revision in this Colab runtime."""
    if importlib.util.find_spec("S4") is not None:
        print("S4 is already importable.")
        return
    run_command(["apt-get", "-qq", "update"])
    run_command([
        "apt-get", "-qq", "install", "-y", "build-essential", "git",
        "libboost-all-dev", "libfftw3-dev", "liblapack-dev",
        "libopenblas-dev", "libsuitesparse-dev",
    ])
    if not S4_DIR.exists():
        run_command(["git", "clone", "https://github.com/phoebe-p/S4.git", str(S4_DIR)])
    run_command(["git", "checkout", S4_COMMIT], cwd=S4_DIR)
    run_command(["make", "-j2", "S4_pyext"], cwd=S4_DIR)
    importlib.invalidate_caches()
    import S4
    print("S4:", S4.__file__)

In [ ]:
RUN_LIVE_OPTICS = True

if RUN_LIVE_OPTICS:
    install_s4()
    run_command([
        sys.executable,
        "validations/validation A.1/run_sodalime_optics_validation.py",
    ], cwd=PROJECT_DIR)
else:
    print("Skipped live optics comparison.")

## 3. Compare thermal and PV parameters

This cell runs the thermal/PV stage directly. The next cell creates the stable report plots used in the repository documentation.

In [ ]:
from radcoolpv import config as config_module
from radcoolpv import pipeline

config_path = PROJECT_DIR / "validations/validation A.1/pv_hemisph_sodalime.yaml"
cfg = config_module.load(str(config_path))
cfg.run.plots = True
context = pipeline.run(cfg)

manifest = json.loads((Path(context.results_dir) / "run.json").read_text())
thermal = context.thermal
published = {
    "Jsc": 355.0,
    "Pmpp": 222.0,
    "Tequil": 319.0,
    "Radiat": 472.0,
    "Voc": 0.722,
}
computed = {
    "Jsc": thermal.isc,
    "Pmpp": thermal.mpp_equil,
    "Tequil": thermal.equil_temp,
    "Radiat": thermal.rad_power_equil,
    "Voc": thermal.voc_equil,
}

rows = ["| Quantity | Published | Computed | Rel. err |", "|---|---:|---:|---:|"]
for key in ["Jsc", "Pmpp", "Tequil", "Radiat", "Voc"]:
    pub = published[key]
    val = computed[key]
    rows.append(f"| {key} | {pub:.3g} | {val:.3f} | {(val - pub) / pub * 100:+.2f}% |")
display(Markdown("\n".join(rows)))

display(Markdown(
    "| Additional PV output | Value |\n|---|---:|\n"
    f"| Vmpp | {thermal.vmpp:.4f} V |\n"
    f"| Fill factor | {thermal.ff_equil:.3f} |\n"
    f"| Efficiency | {thermal.efficiency_equil:.4f} |"
))

print("Results folder:", context.results_dir)

## 4. Generate report plots

In [ ]:
report_args = [
    sys.executable,
    "validations/validation A.1/report_validation.py",
]
if RUN_LIVE_OPTICS:
    report_args.append("--run-live")
run_command(report_args, cwd=PROJECT_DIR)

report_dir = PROJECT_DIR / "validations/validation A.1/report"
display(Markdown((report_dir / "summary.md").read_text()))
for figure in [
    "spectral_comparison.png",
    "band_errors.png",
    "pv_curves.png",
]:
    display(Image(filename=str(report_dir / "figures" / figure)))

## 5. Interpretation

The optical comparison is normal-incidence only because the published reduced spectrum exposes normal-incidence columns but not raw angle-resolved S4 data. The thermal/PV comparison uses the published hemispherical reduced spectrum. That file lacks per-angle emissivity, so the atmospheric term uses the documented angle-independent fallback.

**Exercise:** copy the PV YAML, change `thermal.convection_coefficient` from 12.0 to 8.0 W/m²/K, and predict the direction of the temperature and power changes before running it.

**Reference geometry context:** G. Silva-Oelker and J. Jaramillo-Fernandez (2022), [doi:10.1364/OE.466335](https://doi.org/10.1364/OE.466335).